In [1]:

!python3 -m pip install -U python-woc pandas matplotlib tqdm

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 10.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 11.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 KB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.4/199.4 KB 6.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 12.0 MB/s eta 0:00:0000:0100:01
  Created wheel for python-lzf: filename=python_lzf-0.2.6-cp310-cp310-linux_x86_64.whl size=18236 sha256=e853ae5fbd4738769c60fcbfd85ae95cc583240c0405039064e9eb133140e4d8
  Stored in directory: /home/chasewolverton/.cache/pip/wheels/33/b7/b7/b45ae60d544c15e00be4b2c6c769f209921bfbb07668eac1c4
Successfully built python-lzf
  Attempting uninstall: tqdm
    Found existing installation: tqdm 4.67.3
    Uninstalling tqdm-4.67.3:
      Successfully uninst

In [2]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
# creates the client
woc = WocMapsRemote( base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )

# My 10 assigned WoC projects
projects = [
    "google-research_tensor2robot",
    "graphhopper_jsprit",
    "tdunning_t-digest",
    "villano-lab_nrsicap",
    "matthewheun_byname",
    "fabsig_gpboost",
    "hzwer_arxiv2020-rife",
    "lucmoreau_provtoolbox",
    "autodiff_autodiff",
    "mourner_rbush"
]

list_df_commits = []

for prj in projects:
    print("Getting commits for:", prj)

    commits = woc.get_values("p2c", prj)

    temp_df = pd.DataFrame(commits, columns=["sha1"])
    temp_df["project"] = prj

    list_df_commits.append(temp_df)

df = pd.concat(list_df_commits, ignore_index=True)

# Save so we do not have to retrieve these again
df.to_csv("df_commits.csv", index=False)

print("Total commits:", len(df))
print("Projects returned:", df["project"].nunique())

df.head()

Getting commits for: google-research_tensor2robot
Getting commits for: graphhopper_jsprit
Getting commits for: tdunning_t-digest
Getting commits for: villano-lab_nrsicap
Getting commits for: matthewheun_byname
Getting commits for: fabsig_gpboost
Getting commits for: hzwer_arxiv2020-rife
Getting commits for: lucmoreau_provtoolbox
Getting commits for: autodiff_autodiff
Getting commits for: mourner_rbush
Total commits: 13372
Projects returned: 10


,sha1,project
0,00a4e3109c8ee9c2702639019f620592a6aa9280,google-research_tensor2robot
1,010c934d5ecfbdfa8ae4b5bbca508586e0ff5d3d,google-research_tensor2robot
2,028301b84fe68740472967e12b534cebcb93f450,google-research_tensor2robot
3,02dc7a2ef295368280577b1255d32899a083b5e0,google-research_tensor2robot
4,03307a5ce18a906faf8dff6d1bbb2374edc1e26c,google-research_tensor2robot


In [3]:
print("Number of assigned projects:", len(projects))
print("Number of projects returned:", df["project"].nunique())

print("\nCommits per project:")
print(df.groupby("project").size())

missing = set(projects) - set(df["project"].unique())
print("\nMissing projects:", missing)

Number of assigned projects: 10
Number of projects returned: 10

Commits per project:
project
autodiff_autodiff               1139
fabsig_gpboost                  1256
google-research_tensor2robot     263
graphhopper_jsprit              2951
hzwer_arxiv2020-rife             925
lucmoreau_provtoolbox           3644
matthewheun_byname              1881
mourner_rbush                    409
tdunning_t-digest                581
villano-lab_nrsicap              323
dtype: int64

Missing projects: set()


In [4]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split df['sha1'] in chunks
chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]
commit_data = []

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch',chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(1)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():
    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

  3%|██▍                                                                             | 41/1338 [00:42<22:20,  1.03s/it]

Got Errors {'0bea09152aad94b184cb9d3a1f3b44eefb49bc90': 'Key 0bea09152aad94b184cb9d3a1f3b44eefb49bc90 not found in /da5_fast/All.sha1c/commit_11.tch'}


  4%|███▏                                                                            | 54/1338 [00:56<22:05,  1.03s/it]

Got Errors {'18803ad0ee10e49782e8c017c6481967d64bf05b': 'Key 18803ad0ee10e49782e8c017c6481967d64bf05b not found in /da5_fast/All.sha1c/commit_24.tch'}


  6%|████▋                                                                           | 79/1338 [01:21<21:48,  1.04s/it]

...

Got Errors {'d282b2b32a396e78326e8a3ee21b92cc1e228c4c': 'Key d282b2b32a396e78326e8a3ee21b92cc1e228c4c not found in /da5_fast/All.sha1c/commit_82.tch'}


100%|█████████████████████████████████████████████████████████████████████████████▉| 1337/1338 [23:10<00:01,  1.05s/it]

Got Errors {'fbd509c8f76b73b939a911c1927a4b1c98b12b18': 'Key fbd509c8f76b73b939a911c1927a4b1c98b12b18 not found in /da5_fast/All.sha1c/commit_123.tch'}


100%|██████████████████████████████████████████████████████████████████████████████| 1338/1338 [23:11<00:00,  1.04s/it]


,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,00a4e3109c8ee9c2702639019f620592a6aa9280,ca7894eb679795be0346b330125c73cda538e728,[e65c101b730ad39fc60305eb3d2ab5d5512c6f9b],Tensor2Robot Team <tensor2robot-dev@google.com>,1606260662,-0800,Copybara-Service <copybara-worker@google.com>,1606260698,-0800,Allow predictors to sample from static checkpo...,00a4e3109c8ee9c2702639019f620592a6aa9280,google-research_tensor2robot
1,010c934d5ecfbdfa8ae4b5bbca508586e0ff5d3d,4409b181db88c52109dc8e2d7463fd2e4b1b37a9,[366d97e9d727da7bdc5735f9c5694d433686323a],Tensor2Robot Team <tensor2robot-dev@google.com>,1588007018,-0700,Copybara-Service <copybara-worker@google.com>,1588007052,-0700,Support uint16 in image decoding.\n\nPiperOrig...,010c934d5ecfbdfa8ae4b5bbca508586e0ff5d3d,google-research_tensor2robot


In [6]:
# what does the first record looks like?
# commit  parents
v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

{'commit': '00a4e3109c8ee9c2702639019f620592a6aa9280', 'tree': 'ca7894eb679795be0346b330125c73cda538e728', 'parent': ['e65c101b730ad39fc60305eb3d2ab5d5512c6f9b'], 'author': 'Tensor2Robot Team <tensor2robot-dev@google.com>', 'author_time': 1606260662, 'author_tz': '-0800', 'committer': 'Copybara-Service <copybara-worker@google.com>', 'committer_time': 1606260698, 'committer_tz': '-0800', 'message': 'Allow predictors to sample from static checkpoint.\n\nPiperOrigin-RevId: 344142907\n'}


In [7]:
dfinf = df_commit_data[
    ['project', 'commit', 'author', 'author_time', 'message']
].copy()

dfinf.columns = [
    'project_wocid',
    'commit_sha1',
    'author',
    'time',
    'commit message'
]

dfinf.head()

,project_wocid,commit_sha1,author,time,commit message
0,google-research_tensor2robot,00a4e3109c8ee9c2702639019f620592a6aa9280,Tensor2Robot Team <tensor2robot-dev@google.com>,1606260662,Allow predictors to sample from static checkpo...
1,google-research_tensor2robot,010c934d5ecfbdfa8ae4b5bbca508586e0ff5d3d,Tensor2Robot Team <tensor2robot-dev@google.com>,1588007018,Support uint16 in image decoding.\n\nPiperOrig...
2,google-research_tensor2robot,028301b84fe68740472967e12b534cebcb93f450,Eric Jang <ejang@google.com>,1581106230,Fix broken vrgripper tests.\n\nPiperOrigin-Rev...
3,google-research_tensor2robot,02dc7a2ef295368280577b1255d32899a083b5e0,Tensor2Robot Team <tensor2robot-dev@google.com>,1622141775,Fixes two bugs in default_input_generator.Weig...
4,google-research_tensor2robot,03307a5ce18a906faf8dff6d1bbb2374edc1e26c,Yilei Yang <yileiyang@google.com>,1650047428,Remove unused comments related to Python 2 com...


In [8]:
yournetid = "cwolver1"

dfinf.to_csv(
    yournetid + "_project_summary.csv",
    index=False,
    sep=";"
)

print("Saved:", yournetid + "_project_summary.csv")

Saved: cwolver1_project_summary.csv


In [9]:
print("Total rows:", len(dfinf))
print("Number of projects:", dfinf["project_wocid"].nunique())

print("\nCommits per project:")
print(dfinf.groupby("project_wocid").size())

print("\nDuplicate project/commit rows:")
print(
    dfinf.duplicated(
        subset=["project_wocid", "commit_sha1"]
    ).sum()
)

Total rows: 13295
Number of projects: 10

Commits per project:
project_wocid
autodiff_autodiff               1138
fabsig_gpboost                  1256
google-research_tensor2robot     263
graphhopper_jsprit              2925
hzwer_arxiv2020-rife             925
lucmoreau_provtoolbox           3620
matthewheun_byname              1881
mourner_rbush                    394
tdunning_t-digest                570
villano-lab_nrsicap              323
dtype: int64

Duplicate project/commit rows:
0


In [10]:
woc_stats = (
    dfinf.groupby("project_wocid")
    .agg(
        commits=("commit_sha1", "nunique"),
        authors=("author", "nunique"),
        min_time=("time", "min"),
        max_time=("time", "max")
    )
    .reset_index()
)

woc_stats

,project_wocid,commits,authors,min_time,max_time
0,autodiff_autodiff,1138,70,1531464195,1758747679
1,fabsig_gpboost,1256,17,1585927023,1762259041
2,google-research_tensor2robot,263,25,1547578657,1727687675
3,graphhopper_jsprit,2925,116,1370334347,1761274703
4,hzwer_arxiv2020-rife,925,35,1605159985,1757485923
5,lucmoreau_provtoolbox,3620,33,1314718980,1762536328
6,matthewheun_byname,1881,5,1485710627,1752103929
7,mourner_rbush,394,40,1372862539,1732655528
8,tdunning_t-digest,570,74,1384819635,1744611565
9,villano-lab_nrsicap,323,11,1570638847,1723756861


In [11]:
woc_stats["min_date"] = pd.to_datetime(
    woc_stats["min_time"],
    unit="s"
)

woc_stats["max_date"] = pd.to_datetime(
    woc_stats["max_time"],
    unit="s"
)

woc_stats

,project_wocid,commits,authors,min_time,max_time,min_date,max_date
0,autodiff_autodiff,1138,70,1531464195,1758747679,2018-07-13 06:43:15,2025-09-24 21:01:19
1,fabsig_gpboost,1256,17,1585927023,1762259041,2020-04-03 15:17:03,2025-11-04 12:24:01
2,google-research_tensor2robot,263,25,1547578657,1727687675,2019-01-15 18:57:37,2024-09-30 09:14:35
3,graphhopper_jsprit,2925,116,1370334347,1761274703,2013-06-04 08:25:47,2025-10-24 02:58:23
4,hzwer_arxiv2020-rife,925,35,1605159985,1757485923,2020-11-12 05:46:25,2025-09-10 06:32:03
5,lucmoreau_provtoolbox,3620,33,1314718980,1762536328,2011-08-30 15:43:00,2025-11-07 17:25:28
6,matthewheun_byname,1881,5,1485710627,1752103929,2017-01-29 17:23:47,2025-07-09 23:32:09
7,mourner_rbush,394,40,1372862539,1732655528,2013-07-03 14:42:19,2024-11-26 21:12:08
8,tdunning_t-digest,570,74,1384819635,1744611565,2013-11-19 00:07:15,2025-04-14 06:19:25
9,villano-lab_nrsicap,323,11,1570638847,1723756861,2019-10-09 16:34:07,2024-08-15 21:21:01


In [12]:
github_data = pd.DataFrame([
    ["google-research_tensor2robot", 564, 117, "2024-08-19"],
    ["graphhopper_jsprit", 1800, 642, "2026-09-14"],
    ["tdunning_t-digest", 2200, 232, "2025-02-17"],
    ["villano-lab_nrsicap", 1, 0, "2024-08-15"],
    ["matthewheun_byname", 2, 0, "2026-09-26"],
    ["fabsig_gpboost", 699, 56, "2026-09-28"],
    ["hzwer_arxiv2020-rife", 5600, 570, "2025-09-10"],
    ["lucmoreau_provtoolbox", 83, 43, "2026-07-28"],
    ["autodiff_autodiff", 2000, 196, "2025-01-27"],
    ["mourner_rbush", 2800, 254, "2026-04-03"]
], columns=[
    "project_wocid",
    "stars",
    "forks",
    "last_github_commit"
])

github_data

,project_wocid,stars,forks,last_github_commit
0,google-research_tensor2robot,564,117,2024-08-19
1,graphhopper_jsprit,1800,642,2026-09-14
2,tdunning_t-digest,2200,232,2025-02-17
3,villano-lab_nrsicap,1,0,2024-08-15
4,matthewheun_byname,2,0,2026-09-26
5,fabsig_gpboost,699,56,2026-09-28
6,hzwer_arxiv2020-rife,5600,570,2025-09-10
7,lucmoreau_provtoolbox,83,43,2026-07-28
8,autodiff_autodiff,2000,196,2025-01-27
9,mourner_rbush,2800,254,2026-04-03


In [13]:
part1_stats = woc_stats.merge(
    github_data,
    on="project_wocid",
    how="left"
)

part1_stats

,project_wocid,commits,authors,min_time,max_time,min_date,max_date,stars,forks,last_github_commit
0,autodiff_autodiff,1138,70,1531464195,1758747679,2018-07-13 06:43:15,2025-09-24 21:01:19,2000,196,2025-01-27
1,fabsig_gpboost,1256,17,1585927023,1762259041,2020-04-03 15:17:03,2025-11-04 12:24:01,699,56,2026-09-28
2,google-research_tensor2robot,263,25,1547578657,1727687675,2019-01-15 18:57:37,2024-09-30 09:14:35,564,117,2024-08-19
3,graphhopper_jsprit,2925,116,1370334347,1761274703,2013-06-04 08:25:47,2025-10-24 02:58:23,1800,642,2026-09-14
4,hzwer_arxiv2020-rife,925,35,1605159985,1757485923,2020-11-12 05:46:25,2025-09-10 06:32:03,5600,570,2025-09-10
5,lucmoreau_provtoolbox,3620,33,1314718980,1762536328,2011-08-30 15:43:00,2025-11-07 17:25:28,83,43,2026-07-28
6,matthewheun_byname,1881,5,1485710627,1752103929,2017-01-29 17:23:47,2025-07-09 23:32:09,2,0,2026-09-26
7,mourner_rbush,394,40,1372862539,1732655528,2013-07-03 14:42:19,2024-11-26 21:12:08,2800,254,2026-04-03
8,tdunning_t-digest,570,74,1384819635,1744611565,2013-11-19 00:07:15,2025-04-14 06:19:25,2200,232,2025-02-17
9,villano-lab_nrsicap,323,11,1570638847,1723756861,2019-10-09 16:34:07,2024-08-15 21:21:01,1,0,2024-08-15


# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github